In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR

WindowsPath('c:/Projects/creditlens-ai/data/raw')

In [2]:
bureau = pd.read_csv(
    DATA_DIR / "bureau.csv",
    low_memory=False
)

print(f"Rows: {bureau.shape[0]:,}")
print(f"Columns: {bureau.shape[1]}")
print(
    f"Memory usage: "
    f"{bureau.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB"
)

bureau.head()

Rows: 1,716,428
Columns: 17
Memory usage: 472.82 MB


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [3]:
bureau_customer_summary = pd.Series({
    "bureau_rows": len(bureau),
    "unique_customers": bureau["SK_ID_CURR"].nunique(),
    "unique_bureau_loans": bureau["SK_ID_BUREAU"].nunique(),
    "avg_loans_per_customer": (
        len(bureau) / bureau["SK_ID_CURR"].nunique()
    )
})

bureau_customer_summary

bureau_rows               1.716428e+06
unique_customers          3.058110e+05
unique_bureau_loans       1.716428e+06
avg_loans_per_customer    5.612709e+00
dtype: float64

In [4]:
bureau["CREDIT_ACTIVE"].value_counts(
    dropna=False
)

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

In [5]:
bureau_numeric_features = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),

        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
        BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),

        BUREAU_CREDIT_DAY_OVERDUE_MEAN=(
            "CREDIT_DAY_OVERDUE",
            "mean"
        ),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=(
            "CREDIT_DAY_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_SUM_MEAN=(
            "AMT_CREDIT_SUM",
            "mean"
        ),
        BUREAU_CREDIT_SUM_SUM=(
            "AMT_CREDIT_SUM",
            "sum"
        ),
        BUREAU_CREDIT_SUM_MAX=(
            "AMT_CREDIT_SUM",
            "max"
        ),

        BUREAU_DEBT_MEAN=(
            "AMT_CREDIT_SUM_DEBT",
            "mean"
        ),
        BUREAU_DEBT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "sum"
        ),
        BUREAU_DEBT_MAX=(
            "AMT_CREDIT_SUM_DEBT",
            "max"
        ),

        BUREAU_OVERDUE_SUM=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum"
        ),
        BUREAU_OVERDUE_MAX=(
            "AMT_CREDIT_SUM_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_PROLONG_SUM=(
            "CNT_CREDIT_PROLONG",
            "sum"
        )
    )
    .reset_index()
)

bureau_numeric_features.head()

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_SUM,BUREAU_CREDIT_SUM_MAX,BUREAU_DEBT_MEAN,BUREAU_DEBT_SUM,BUREAU_DEBT_MAX,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_CREDIT_PROLONG_SUM
0,100001,7,-735.000000,-1572,-49,0.0,0,207623.571429,1453365.000,378000.0,85240.928571,596686.5,373239.0,0.0,0.0,0
1,100002,8,-874.000000,-1437,-103,0.0,0,108131.945625,865055.565,450000.0,49156.200000,245781.0,245781.0,0.0,0.0,0
2,100003,4,-1400.750000,-2586,-606,0.0,0,254350.125000,1017400.500,810000.0,0.000000,0.0,0.0,0.0,0.0,0
3,100004,2,-867.000000,-1326,-408,0.0,0,94518.900000,189037.800,94537.8,0.000000,0.0,0.0,0.0,0.0,0
4,100005,3,-190.666667,-373,-62,0.0,0,219042.000000,657126.000,568800.0,189469.500000,568408.5,543087.0,0.0,0.0,0


In [6]:
bureau_numeric_features = (
    bureau
    .groupby("SK_ID_CURR")
    .agg(
        BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),

        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
        BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),

        BUREAU_CREDIT_DAY_OVERDUE_MEAN=(
            "CREDIT_DAY_OVERDUE",
            "mean"
        ),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=(
            "CREDIT_DAY_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_SUM_MEAN=(
            "AMT_CREDIT_SUM",
            "mean"
        ),
        BUREAU_CREDIT_SUM_SUM=(
            "AMT_CREDIT_SUM",
            "sum"
        ),
        BUREAU_CREDIT_SUM_MAX=(
            "AMT_CREDIT_SUM",
            "max"
        ),

        BUREAU_DEBT_MEAN=(
            "AMT_CREDIT_SUM_DEBT",
            "mean"
        ),
        BUREAU_DEBT_SUM=(
            "AMT_CREDIT_SUM_DEBT",
            "sum"
        ),
        BUREAU_DEBT_MAX=(
            "AMT_CREDIT_SUM_DEBT",
            "max"
        ),

        BUREAU_OVERDUE_SUM=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum"
        ),
        BUREAU_OVERDUE_MAX=(
            "AMT_CREDIT_SUM_OVERDUE",
            "max"
        ),

        BUREAU_CREDIT_PROLONG_SUM=(
            "CNT_CREDIT_PROLONG",
            "sum"
        )
    )
    .reset_index()
)

bureau_numeric_features.head()

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_SUM,BUREAU_CREDIT_SUM_MAX,BUREAU_DEBT_MEAN,BUREAU_DEBT_SUM,BUREAU_DEBT_MAX,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_CREDIT_PROLONG_SUM
0,100001,7,-735.000000,-1572,-49,0.0,0,207623.571429,1453365.000,378000.0,85240.928571,596686.5,373239.0,0.0,0.0,0
1,100002,8,-874.000000,-1437,-103,0.0,0,108131.945625,865055.565,450000.0,49156.200000,245781.0,245781.0,0.0,0.0,0
2,100003,4,-1400.750000,-2586,-606,0.0,0,254350.125000,1017400.500,810000.0,0.000000,0.0,0.0,0.0,0.0,0
3,100004,2,-867.000000,-1326,-408,0.0,0,94518.900000,189037.800,94537.8,0.000000,0.0,0.0,0.0,0.0,0
4,100005,3,-190.666667,-373,-62,0.0,0,219042.000000,657126.000,568800.0,189469.500000,568408.5,543087.0,0.0,0.0,0


In [7]:
credit_status_counts = (
    pd.crosstab(
        bureau["SK_ID_CURR"],
        bureau["CREDIT_ACTIVE"]
    )
    .add_prefix("BUREAU_STATUS_")
    .reset_index()
)

credit_status_counts.head()

CREDIT_ACTIVE,SK_ID_CURR,BUREAU_STATUS_Active,BUREAU_STATUS_Bad debt,BUREAU_STATUS_Closed,BUREAU_STATUS_Sold
0,100001,3,0,4,0
1,100002,2,0,6,0
2,100003,1,0,3,0
3,100004,0,0,2,0
4,100005,2,0,1,0


In [8]:
bureau_features = bureau_numeric_features.merge(
    credit_status_counts,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

print(f"Rows: {bureau_features.shape[0]:,}")
print(f"Columns: {bureau_features.shape[1]}")

bureau_features.head()

Rows: 305,811
Columns: 20


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_SUM_MEAN,BUREAU_CREDIT_SUM_SUM,BUREAU_CREDIT_SUM_MAX,BUREAU_DEBT_MEAN,BUREAU_DEBT_SUM,BUREAU_DEBT_MAX,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_CREDIT_PROLONG_SUM,BUREAU_STATUS_Active,BUREAU_STATUS_Bad debt,BUREAU_STATUS_Closed,BUREAU_STATUS_Sold
0,100001,7,-735.000000,-1572,-49,0.0,0,207623.571429,1453365.000,378000.0,85240.928571,596686.5,373239.0,0.0,0.0,0,3,0,4,0
1,100002,8,-874.000000,-1437,-103,0.0,0,108131.945625,865055.565,450000.0,49156.200000,245781.0,245781.0,0.0,0.0,0,2,0,6,0
2,100003,4,-1400.750000,-2586,-606,0.0,0,254350.125000,1017400.500,810000.0,0.000000,0.0,0.0,0.0,0.0,0,1,0,3,0
3,100004,2,-867.000000,-1326,-408,0.0,0,94518.900000,189037.800,94537.8,0.000000,0.0,0.0,0.0,0.0,0,0,0,2,0
4,100005,3,-190.666667,-373,-62,0.0,0,219042.000000,657126.000,568800.0,189469.500000,568408.5,543087.0,0.0,0.0,0,2,0,1,0


In [9]:
print(
    "Unique customers:",
    bureau_features["SK_ID_CURR"].nunique()
)

print(
    "SK_ID_CURR unique:",
    bureau_features["SK_ID_CURR"].is_unique
)

Unique customers: 305811
SK_ID_CURR unique: True


In [10]:
bureau_features_path = (
    INTERIM_DIR / "bureau_customer_features.csv"
)

bureau_features.to_csv(
    bureau_features_path,
    index=False
)

print(f"Saved to: {bureau_features_path}")

Saved to: c:\Projects\creditlens-ai\data\interim\bureau_customer_features.csv


In [11]:
import gc

del bureau
del bureau_numeric_features
del credit_status_counts

gc.collect()

print("Raw bureau objects removed from memory.")

Raw bureau objects removed from memory.


In [12]:
application_train_ids = pd.read_csv(
    DATA_DIR / "application_train.csv",
    usecols=["SK_ID_CURR", "TARGET"]
)

bureau_coverage = application_train_ids.merge(
    bureau_features[["SK_ID_CURR"]],
    on="SK_ID_CURR",
    how="left",
    indicator=True,
    validate="one_to_one"
)

coverage_summary = (
    bureau_coverage["_merge"]
    .value_counts()
)

coverage_summary

_merge
both          263491
left_only      44020
right_only         0
Name: count, dtype: int64

In [13]:
matched_customers = (
    bureau_coverage["_merge"] == "both"
).sum()

total_customers = len(bureau_coverage)

print(
    f"Customers with bureau history: "
    f"{matched_customers:,}"
)

print(
    f"Coverage: "
    f"{matched_customers / total_customers * 100:.2f}%"
)

Customers with bureau history: 263,491
Coverage: 85.69%


In [14]:
application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

train_with_bureau = application_train.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

print(
    f"Original shape: "
    f"{application_train.shape}"
)

print(
    f"With bureau shape: "
    f"{train_with_bureau.shape}"
)

Original shape: (307511, 122)
With bureau shape: (307511, 141)


In [15]:
train_with_bureau["BUREAU_HAS_HISTORY"] = (
    train_with_bureau["BUREAU_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

count_columns = [
    "BUREAU_LOAN_COUNT",
    "BUREAU_STATUS_Active",
    "BUREAU_STATUS_Bad debt",
    "BUREAU_STATUS_Closed",
    "BUREAU_STATUS_Sold"
]

train_with_bureau[count_columns] = (
    train_with_bureau[count_columns]
    .fillna(0)
)

train_with_bureau[
    [
        "SK_ID_CURR",
        "BUREAU_HAS_HISTORY",
        *count_columns
    ]
].head(10)

C:\Users\Vıctus\AppData\Local\Temp\ipykernel_23348\1847904526.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_with_bureau["BUREAU_HAS_HISTORY"] = (


,SK_ID_CURR,BUREAU_HAS_HISTORY,BUREAU_LOAN_COUNT,BUREAU_STATUS_Active,BUREAU_STATUS_Bad debt,BUREAU_STATUS_Closed,BUREAU_STATUS_Sold
0,100002,1,8.0,2.0,0.0,6.0,0.0
1,100003,1,4.0,1.0,0.0,3.0,0.0
2,100004,1,2.0,0.0,0.0,2.0,0.0
3,100006,0,0.0,0.0,0.0,0.0,0.0
4,100007,1,1.0,0.0,0.0,1.0,0.0
5,100008,1,3.0,1.0,0.0,2.0,0.0
6,100009,1,18.0,4.0,0.0,14.0,0.0
7,100010,1,2.0,1.0,0.0,1.0,0.0
8,100011,1,4.0,0.0,0.0,4.0,0.0
9,100012,0,0.0,0.0,0.0,0.0,0.0


In [16]:
# Fragmentation uyarısını temizlemek ve bellekte
# daha düzenli bir DataFrame oluşturmak için.
train_with_bureau = train_with_bureau.copy()

print(f"Final train + bureau shape: {train_with_bureau.shape}")

Final train + bureau shape: (307511, 142)


In [17]:
del application_train
del application_train_ids
del bureau_coverage

gc.collect()

print("Unused intermediate objects removed.")

Unused intermediate objects removed.


In [18]:
y_bureau = (
    train_with_bureau["TARGET"]
    .astype("int8")
)

X_bureau = (
    train_with_bureau
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)

X_bureau["DAYS_EMPLOYED_ANOMALY"] = (
    X_bureau["DAYS_EMPLOYED"] == 365243
).astype("int8")

X_bureau.loc[
    X_bureau["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(f"X shape: {X_bureau.shape}")
print(f"y shape: {y_bureau.shape}")

X shape: (307511, 141)
y shape: (307511,)


In [19]:
from sklearn.model_selection import train_test_split

X_train_bureau, X_val_bureau, y_train_bureau, y_val_bureau = (
    train_test_split(
        X_bureau,
        y_bureau,
        test_size=0.20,
        random_state=42,
        stratify=y_bureau
    )
)

print(f"Train: {X_train_bureau.shape}")
print(f"Validation: {X_val_bureau.shape}")

print(
    f"Train positive rate: "
    f"{y_train_bureau.mean() * 100:.2f}%"
)

print(
    f"Validation positive rate: "
    f"{y_val_bureau.mean() * 100:.2f}%"
)

Train: (246008, 141)
Validation: (61503, 141)
Train positive rate: 8.07%
Validation positive rate: 8.07%


In [20]:
numeric_columns_bureau = (
    X_train_bureau
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

categorical_columns_bureau = (
    X_train_bureau
    .select_dtypes(exclude=["number"])
    .columns
    .tolist()
)

print(
    f"Numeric features: "
    f"{len(numeric_columns_bureau)}"
)

print(
    f"Categorical features: "
    f"{len(categorical_columns_bureau)}"
)

print(
    f"Total: "
    f"{len(numeric_columns_bureau) + len(categorical_columns_bureau)}"
)

Numeric features: 125
Categorical features: 16
Total: 141


In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [22]:
numeric_pipeline_bureau = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline_bureau = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor_bureau = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_bureau,
            numeric_columns_bureau
        ),
        (
            "categorical",
            categorical_pipeline_bureau,
            categorical_columns_bureau
        )
    ],
    sparse_threshold=1.0
)

In [23]:
import time

bureau_balanced_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_bureau
        ),
        (
            "classifier",
            LogisticRegression(
                solver="saga",
                max_iter=1500,
                tol=1e-3,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

start_time = time.time()

bureau_balanced_model.fit(
    X_train_bureau,
    y_train_bureau
)

bureau_training_time = (
    time.time() - start_time
)

bureau_classifier = (
    bureau_balanced_model
    .named_steps["classifier"]
)

print(
    f"Training time: "
    f"{bureau_training_time:.2f} seconds"
)

print(
    f"Iterations used: "
    f"{bureau_classifier.n_iter_[0]}"
)

Training time: 123.40 seconds
Iterations used: 369


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [25]:
bureau_probabilities = (
    bureau_balanced_model.predict_proba(
        X_val_bureau
    )[:, 1]
)

bureau_predictions = (
    bureau_probabilities >= 0.50
).astype(int)

bureau_metrics = {
    "accuracy": accuracy_score(
        y_val_bureau,
        bureau_predictions
    ),
    "precision": precision_score(
        y_val_bureau,
        bureau_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_val_bureau,
        bureau_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_val_bureau,
        bureau_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_val_bureau,
        bureau_probabilities
    ),
    "pr_auc": average_precision_score(
        y_val_bureau,
        bureau_probabilities
    )
}

pd.Series(bureau_metrics)

accuracy     0.691348
precision    0.163353
recall       0.684995
f1           0.263797
roc_auc      0.753412
pr_auc       0.235836
dtype: float64

In [26]:
bureau_confusion_matrix = confusion_matrix(
    y_val_bureau,
    bureau_predictions
)

bureau_confusion_matrix

array([[39119, 17419],
       [ 1564,  3401]])

In [27]:
baseline_balanced_metrics = {
    "accuracy": 0.689300,
    "precision": 0.161368,
    "recall": 0.678751,
    "f1": 0.260745,
    "roc_auc": 0.748826,
    "pr_auc": 0.228761
}

bureau_comparison = pd.DataFrame({
    "Application_Only": baseline_balanced_metrics,
    "Application_Plus_Bureau": bureau_metrics
}).T

bureau_comparison

,accuracy,precision,recall,f1,roc_auc,pr_auc
Application_Only,0.689300,0.161368,0.678751,0.260745,0.748826,0.228761
Application_Plus_Bureau,0.691348,0.163353,0.684995,0.263797,0.753412,0.235836


In [28]:
metric_improvement = (
    bureau_comparison
    .loc["Application_Plus_Bureau"]
    -
    bureau_comparison
    .loc["Application_Only"]
)

metric_improvement

accuracy     0.002048
precision    0.001985
recall       0.006244
f1           0.003052
roc_auc      0.004586
pr_auc       0.007075
dtype: float64

In [29]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

bureau_comparison_path = (
    REPORTS_DIR / "bureau_feature_experiment.csv"
)

bureau_comparison.to_csv(
    bureau_comparison_path,
    index=True
)

print(f"Saved to: {bureau_comparison_path}")

Saved to: c:\Projects\creditlens-ai\reports\bureau_feature_experiment.csv


In [30]:
bureau_confusion_results = pd.DataFrame({
    "model": [
        "Application_Only_Balanced",
        "Application_Plus_Bureau_Balanced"
    ],
    "TN": [
        39024,
        bureau_confusion_matrix[0, 0]
    ],
    "FP": [
        17514,
        bureau_confusion_matrix[0, 1]
    ],
    "FN": [
        1595,
        bureau_confusion_matrix[1, 0]
    ],
    "TP": [
        3370,
        bureau_confusion_matrix[1, 1]
    ]
})

bureau_confusion_results.to_csv(
    REPORTS_DIR / "bureau_confusion_comparison.csv",
    index=False
)

bureau_confusion_results

,model,TN,FP,FN,TP
0,Application_Only_Balanced,39024,17514,1595,3370
1,Application_Plus_Bureau_Balanced,39119,17419,1564,3401


## Bureau Feature Engineering Result

Customer-level historical credit features were engineered from approximately 1.7 million bureau records and joined to the application dataset.

Bureau information was available for 85.69% of training applicants.

Adding bureau-derived features improved the balanced Logistic Regression validation performance:

- ROC-AUC: 0.7488 → 0.7534
- PR-AUC: 0.2288 → 0.2358
- Recall: 0.6788 → 0.6850
- F1: 0.2607 → 0.2638

The experiment indicates that historical credit behavior contains additional predictive information beyond the application-level features.